<a href="https://colab.research.google.com/github/OPIM5512-aya16102/myscrapers-aya16102/blob/main/Model_Trending.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 📊 Welcome to the Craigslist - Car Price Predictor Notebook
------------
**Ahmed Yosif Ahmed - University of Connecticut**

This is where you can monitor the models' performances! Three prediction models were used to predict the price of vehicles that were scraped from the website using GenAI .

---
**🧱 1. Shared Feature Engineering (All Models)**
------------------------------------------------------------
**Data Cleaning:**
- Converted 'price' and 'mileage' to numeric values
- Removed rows with missing or non-positive prices
- Standardized 'zipcode' to 5-digit string format

**Feature Creation:**
- 'make_model' = make + model (captures interaction)
- 'age' = 2026 - year (proxy for depreciation)

**Feature Types:**



1.   Categorical: color, condition, transmission, fuel,city,state,zip-code
2.   Numeric: age, miles|



**Preprocessing Pipeline (ColumnTransformer):**

1. High-cardinality handling (make_model):
    - Top-K encoding (Top 15 most frequent categories)
    - Remaining grouped into "other"
    - OneHotEncoding applied

2. Other categorical features:
    - Missing values → most frequent
    - OneHotEncoding (handle_unknown="ignore")

3. Numerical features:
    - Missing values → median imputation

------------------------------------------------------------
🌳 2. Model-Specific Approaches
------------------------------------------------------------

1. Decision Tree (DecisionTreeRegressor):
    
    Transformation:

    *   No target transformation
    *   Trained directly on raw price



    Tuned Hyperparameters:

    *   max_depth
    *   min_samples_leaf

    Characteristics:
    
    *   Simple, interpretable
    *   High variance (can overfit)


2. Random Forest (RandomForestRegressor):

    Transformation:
    *    Train: log10(price)
    *    Predict: inverse (10^x)

    Tuned Hyperparameters:
      *    n_estimators
      *    max_depth
      *    min_samples_leaf

    Characteristics:
      *    Bagging ensemble
      *    Reduces variance vs Decision Tree
      *    More stable predictions


3. XGBoost (XGBRegressor):
    Transformation:
    *    Train: log10(price)
    *    Predict: inverse (10^x)

    Tuned Hyperparameters:
    *    n_estimators
    *    max_depth
    *    learning_rate
    *    subsample

    Characteristics:
    *    Gradient boosting (sequential learning)
    *    Captures complex nonlinear relationships
    *    Typically best performing model

------------------------------------------------------------
🔁 3. Training Strategy
------------------------------------------------------------

Time-based Split:
- Training: all data before latest date
- Holdout: latest date only

Purpose:
- Simulates real-world production predictions
- Prevents data leakage

Hyperparameter Tuning:
- RandomizedSearchCV
- 3-fold cross-validation (KFold)
- Scoring metric: Mean Absolute Error (MAE), MAPE, RMSE, R2, and Bias

Model Selection:
- Best hyperparameters selected via CV
- Model refit on full training dataset

Evaluation:
- Performance metrics of models in real time.
- Permutation Importance and Partial Dependance Regression (PDP) plots for top 3 features.

Interpreting Feature Importance & PDP Plots:
- Feature Importance: Impact of features on price in testing data.
- PDP: Effects of top 3 features on price in training data.




------------------------------------------------------------
⚠️ **Ensure you are connected to a Kernel. In the taskbar, select "Runtime" -> "Run All"**

⚠️**To re-sync with repo: select "Runtime" -> "Disconnect and delete runtime" -> Reconnect to Kernel and repeat above.**

------------------------------------------------------------

In [1]:
jupyter nbconvert --clear-output --inplace your_notebook.ipynb
# @title
!pip install -q ipywidgets
# clone repository
!git clone https://github.com/OPIM5512-aya16102/myscrapers-aya16102.git

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

BASE_DIR = "/content/myscrapers-aya16102/results"

# 1. Load and Calculate Data from CSVs
run_folders = sorted([f for f in os.listdir(BASE_DIR) if f.isdigit()])

metrics_data = []
importance_data = []

for ts in run_folders:
    dt_obj = pd.to_datetime(ts, format='%Y%m%d%H')

    # Look dynamically for available models in this run's preds folder
    preds_dir = os.path.join(BASE_DIR, ts, "preds")
    if os.path.exists(preds_dir):
        for file in os.listdir(preds_dir):
            if file.endswith("_preds.csv"):
                model = file.replace("_preds.csv", "")

                # Calculate Metrics
                df_preds = pd.read_csv(os.path.join(preds_dir, file))
                y_true = df_preds["actual"].values
                y_pred = df_preds["pred"].values

                mae = mean_absolute_error(y_true, y_pred)
                rmse = np.sqrt(mean_squared_error(y_true, y_pred))

                mask = y_true != 0
                mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan

                r2 = r2_score(y_true, y_pred)
                bias = np.mean(y_pred - y_true)

                metrics_data.append({
                    "timestamp": dt_obj,
                    "run_id": ts,
                    "model": model,
                    "mae": mae,
                    "rmse": rmse,
                    "mape": mape,
                    "r2": r2,
                    "bias": bias
                })

                # Extract Feature Importance Data
                err_file = os.path.join(BASE_DIR, ts, "errors", f"{model}_perm_importance.csv")
                if os.path.exists(err_file):
                    df_err = pd.read_csv(err_file)
                    for _, row in df_err.iterrows():
                        importance_data.append({
                            "timestamp": dt_obj,
                            "model": model,
                            "feature": row["feature"],
                            "importance_mean": row["importance_mean"]
                        })

df_metrics = pd.DataFrame(metrics_data)
df_importance = pd.DataFrame(importance_data)

if df_metrics.empty:
    print("No metric data found. Make sure runs are synced to the 'results' folder.")
else:
    models_available = sorted(df_metrics['model'].unique())
def render_dashboard(df_metrics, df_importance):
    if df_metrics.empty:
        print("No metric data found.")
        return

    models_available = sorted(df_metrics['model'].unique())

    model_dropdown = widgets.Dropdown(
        options=models_available,
        description='Target Model:',
        style={'description_width': 'initial'}
    )

    output_plot = widgets.Output()

    def update_trend_plots(*args):
        ...

    model_dropdown.observe(update_trend_plots, names='value')
    display(model_dropdown, output_plot)
    update_trend_plots()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 15.0 MB/s eta 0:00:00
Cloning into 'myscrapers-aya16102'...
remote: Enumerating objects: 1891, done.
remote: Counting objects: 100% (362/362), done.
remote: Compressing objects: 100% (311/311), done.
remote: Total 1891 (delta 81), reused 155 (delta 44), pack-reused 1529 (from 1)
Receiving objects: 100% (1891/1891), 9.18 MiB | 12.77 MiB/s, done.
Resolving deltas: 100% (811/811), done.


In [2]:
# @title 📊 Interactive Stakeholder Dashboard
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import warnings
warnings.filterwarnings('ignore')

def render_performance_dashboard(base_dir="/content/myscrapers-aya16102/results"):
    """
    Scans the results directory for historical model predictions and feature importance,
    calculates metrics, and renders an interactive dashboard for stakeholders.
    """

    # ---------------------------------------------------------
    # 1. LOAD AND AGGREGATE DATA
    # ---------------------------------------------------------
    if not os.path.exists(base_dir):
        display(HTML(f"<b style='color:red;'>Error:</b> Directory not found: <code>{base_dir}</code>"))
        return

    run_folders = sorted([f for f in os.listdir(base_dir) if f.isdigit()])
    metrics_data = []
    importance_data = []

    for ts in run_folders:
        dt_obj = pd.to_datetime(ts, format='%Y%m%d%H')
        preds_dir = os.path.join(base_dir, ts, "preds")

        if os.path.exists(preds_dir):
            for file in os.listdir(preds_dir):
                if file.endswith("_preds.csv"):
                    model = file.replace("_preds.csv", "")

                    # Calculate Metrics
                    try:
                        df_preds = pd.read_csv(os.path.join(preds_dir, file))
                        y_true = df_preds["actual"].values
                        y_pred = df_preds["pred"].values

                        mae = mean_absolute_error(y_true, y_pred)
                        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
                        mask = y_true != 0
                        mape = np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100 if mask.any() else np.nan
                        r2 = r2_score(y_true, y_pred)
                        bias = np.mean(y_pred - y_true)

                        metrics_data.append({
                            "timestamp": dt_obj, "run_id": ts, "model": model,
                            "mae": mae, "rmse": rmse, "mape": mape, "r2": r2, "bias": bias
                        })
                    except Exception:
                        continue # Skip corrupted files silently

                    # Extract Feature Importance Data
                    err_file = os.path.join(base_dir, ts, "errors", f"{model}_perm_importance.csv")
                    if os.path.exists(err_file):
                        try:
                            df_err = pd.read_csv(err_file)
                            for _, row in df_err.iterrows():
                                importance_data.append({
                                    "timestamp": dt_obj, "model": model,
                                    "feature": row["feature"], "importance_mean": row["importance_mean"]
                                })
                        except Exception:
                            continue

    df_metrics = pd.DataFrame(metrics_data)
    df_importance = pd.DataFrame(importance_data)

    if df_metrics.empty:
        display(HTML("<b style='color:red;'>No metric data found.</b> Make sure GitHub Actions has synced the 'results' folder."))
        return

    # ---------------------------------------------------------
    # 2. BUILD DASHBOARD UI
    # ---------------------------------------------------------
    models_available = sorted(df_metrics['model'].unique())

    # Header & Metric Guide for Stakeholders
    header_html = HTML("""
    <div style="font-family: Arial, sans-serif; margin-bottom: 20px; padding: 15px; background-color: #f8f9fa; border-radius: 8px; border-left: 5px solid #2c3e50;">
        <h2 style="margin-top: 0; color: #2c3e50;">📈 Model Performance & Trends Dashboard</h2>
        <p style="margin-bottom: 5px;"><b style="color: black;">Select a modelfrom the dropdown below to view its historical accuracy and feature stability.</p>
        <ul style="margin-top: 5px; font-size: 13px; color: #555;">
            <li><b>MAE / RMSE / MAPE:</b> Lower is better. Measures the average error in predicted price.</li>
            <li><b>R² (R-Squared):</b> Higher is better (Max 1.0). Measures how well the model explains price variance.</li>
            <li><b>Bias:</b> Ideally 0. Positive means the model overprices; negative means it underprices.</li>
        </ul>
    </div>
    """)

    model_dropdown = widgets.Dropdown(
        options=[(m.upper(), m) for m in models_available],
        description='<b>Target Model:</b>',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='300px', margin='0 0 20px 0')
    )

    output_plot = widgets.Output()

    # ---------------------------------------------------------
    # 3. PLOTTING LOGIC
    # ---------------------------------------------------------
    def update_trend_plots(*args):
        selected_model = model_dropdown.value

        with output_plot:
            clear_output(wait=True)
            sns.set_theme(style="whitegrid", rc={"axes.spines.right": False, "axes.spines.top": False})

            fig, axes = plt.subplots(2, 3, figsize=(18, 11))
            fig.suptitle(f"Historical Trends: {selected_model.upper()} Model", fontsize=20, fontweight='bold', y=0.98)
            axes = axes.flatten()

            model_metrics = df_metrics[df_metrics['model'] == selected_model]

            # Extract EXACT unique timestamps that exist in the data to fix the X-axis
            unique_timestamps = model_metrics['timestamp'].unique()

            metrics = [
                ("mae", "Mean Abs Error ($)", "Lower is Better", "#e74c3c"),
                ("rmse", "Root Mean Sq Error ($)", "Lower is Better", "#e67e22"),
                ("mape", "Mean Abs % Error", "Lower is Better", "#f1c40f"),
                ("r2", "R² Score", "Higher is Better", "#2ecc71"),
                ("bias", "Prediction Bias ($)", "Closer to 0 is Better", "#9b59b6")
            ]

            # --- Plot Metrics (Grid indices 0 to 4) ---
            for i, (col, title, direction, color) in enumerate(metrics):
                sns.lineplot(
                    data=model_metrics, x="timestamp", y=col,
                    marker="o", linewidth=2.5, markersize=8, color=color, ax=axes[i]
                )
                axes[i].set_title(f"{title}\n({direction})", fontsize=13, pad=10)
                axes[i].set_xlabel("")
                axes[i].set_ylabel("")

                # Force x-axis to ONLY tick on the actual dates that exist
                axes[i].set_xticks(unique_timestamps)
                # FIX: Removed the % sign from %00
                axes[i].xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:00'))

                # Rotated 45 degrees so they don't overlap if you have many runs
                axes[i].tick_params(axis='x', rotation=45)

                if col == "bias":
                    axes[i].axhline(0, color='gray', linestyle='--', alpha=0.7)

            # --- Plot Feature Importance Trend (Grid index 5) ---
            model_imp = df_importance[df_importance['model'] == selected_model]

            if not model_imp.empty:
                latest_time = model_imp['timestamp'].max()
                latest_imp = model_imp[model_imp['timestamp'] == latest_time]
                top_features = latest_imp.nlargest(5, 'importance_mean')['feature'].tolist()
                plot_imp = model_imp[model_imp['feature'].isin(top_features)]

                sns.lineplot(
                    data=plot_imp, x="timestamp", y="importance_mean",
                    hue="feature", marker="s", linewidth=2.5, markersize=7,
                    palette="tab10", ax=axes[5]
                )
                axes[5].set_title("Top 5 Drivers of Price\n(Feature Stability Over Time)", fontsize=13, pad=10)
                axes[5].set_xlabel("")
                axes[5].set_ylabel("Impact on Error")

                # Force x-axis to ONLY tick on the actual dates that exist
                imp_unique_timestamps = plot_imp['timestamp'].unique()
                axes[5].set_xticks(imp_unique_timestamps)
                # FIX: Removed the % sign from %00
                axes[5].xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:00'))
                axes[5].tick_params(axis='x', rotation=45)

                axes[5].legend(title="Features", bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
            else:
                axes[5].text(0.5, 0.5, "No Feature Importance Data", ha='center', va='center', fontsize=12, color="gray")
                axes[5].set_axis_off()

            plt.tight_layout(rect=[0, 0, 1, 0.95])
            plt.show()

    # 4. Bind and Display
    model_dropdown.observe(update_trend_plots, names='value')

    dashboard_ui = widgets.VBox([
        widgets.HTML(header_html.data),
        model_dropdown,
        output_plot
    ])

    display(dashboard_ui)
    update_trend_plots()

# ==========================================
# Run the dashboard
# ==========================================
render_performance_dashboard()

In [10]:
# @title 📊 Comprehensive Feature Importance & Error Dashboard
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
import warnings
warnings.filterwarnings('ignore')

def render_all_features_error_dashboard(base_dir="/content/myscrapers-aya16102/results"):
    """
    Scans the results directory for permutation importance data,
    rendering a dual-view dashboard: ALL features (current) + Time Trend (Selected feature).
    """

    # ---------------------------------------------------------
    # 1. LOAD AND AGGREGATE DATA
    # ---------------------------------------------------------
    if not os.path.exists(base_dir):
        display(HTML(f"<b style='color:red;'>Error:</b> Directory not found: <code>{base_dir}</code>"))
        return

    run_folders = sorted([f for f in os.listdir(base_dir) if f.isdigit()])
    importance_data = []

    for ts in run_folders:
        dt_obj = pd.to_datetime(ts, format='%Y%m%d%H')
        err_dir = os.path.join(base_dir, ts, "errors")

        if os.path.exists(err_dir):
            for file in os.listdir(err_dir):
                if file.endswith("_perm_importance.csv"):
                    model = file.replace("_perm_importance.csv", "")

                    try:
                        df_err = pd.read_csv(os.path.join(err_dir, file))
                        for _, row in df_err.iterrows():
                            importance_data.append({
                                "timestamp": dt_obj,
                                "run_id": ts,
                                "model": model,
                                "feature": row["feature"],
                                "importance_mean": row["importance_mean"],
                                "importance_std": row["importance_std"]
                            })
                    except Exception:
                        continue

    df_imp = pd.DataFrame(importance_data)

    if df_imp.empty:
        display(HTML("<b style='color:red;'>No feature importance data found.</b> Make sure the 'errors' folder is synced."))
        return

    # ---------------------------------------------------------
    # 2. BUILD DASHBOARD UI
    # ---------------------------------------------------------
    models_available = sorted(df_imp['model'].unique())
    initial_model = models_available[0]

    # Get initial features for the first model (sorted by latest importance)
    latest_ts = df_imp[df_imp['model'] == initial_model]['timestamp'].max()
    initial_feats = df_imp[(df_imp['model'] == initial_model) & (df_imp['timestamp'] == latest_ts)]
    initial_feats = initial_feats.sort_values('importance_mean', ascending=False)['feature'].tolist()

    # Header & Guide (Clean White Card Style)
    header_html = HTML("""
    <div style="font-family: Arial, sans-serif; margin-bottom: 20px; padding: 20px; background-color: #ffffff; border-radius: 8px; border: 1px solid #eaeaea; box-shadow: 0 4px 6px rgba(0,0,0,0.04); border-left: 5px solid #16a085;">
        <h2 style="margin-top: 0; color: #16a085;">🔎 Feature Importance Deep-Dive Dashboard</h2>
        <p style="margin-bottom: 5px;">View the relative importance of <b>ALL</b> features, and track how a specific feature's reliability changes over time.</p>
        <ul style="margin-top: 5px; font-size: 13px; color: #555;">
            <li><b>Left Chart:</b> The latest snapshot of all features. The black error bars represent the Standard Deviation (uncertainty).</li>
            <li><b>Right Chart:</b> The historical trend for the feature selected in the dropdown. The shaded region is the Error Band (Mean ± 1 Std Dev).</li>
        </ul>
    </div>
    """)

    model_dropdown = widgets.Dropdown(
        options=[(m.upper(), m) for m in models_available],
        value=initial_model,
        description='<b>Model:</b>',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='250px')
    )

    feature_dropdown = widgets.Dropdown(
        options=initial_feats,
        value=initial_feats[0] if initial_feats else None,
        description='<b>Track Feature:</b>',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='350px')
    )

    output_plot = widgets.Output()

    # ---------------------------------------------------------
    # 3. PLOTTING LOGIC
    # ---------------------------------------------------------
    def update_plots(*args):
        selected_model = model_dropdown.value
        selected_feature = feature_dropdown.value

        with output_plot:
            clear_output(wait=True)

            # Clean White Theme
            sns.set_theme(
                style="white",
                rc={
                    "axes.facecolor": "#ffffff", "figure.facecolor": "#ffffff",
                    "axes.spines.right": False, "axes.spines.top": False,
                    "axes.grid": True, "grid.color": "#eeeeee"
                }
            )

            fig, axes = plt.subplots(1, 2, figsize=(18, 8))
            fig.patch.set_facecolor('#ffffff')

            model_data = df_imp[df_imp['model'] == selected_model]

            if not model_data.empty:
                # --- PLOT 1: ALL FEATURES (LATEST RUN) ---
                latest_time = model_data['timestamp'].max()
                latest_data = model_data[model_data['timestamp'] == latest_time].copy()

                # Cap at Top 25 features so the Y-axis text doesn't become a black smudge
                latest_data = latest_data.sort_values('importance_mean', ascending=True).tail(25)

                formatted_latest = latest_time.strftime('%b %d, %Y - %H:00')
                axes[0].set_title(f"All Features & Error Bounds\n(Snapshot: {formatted_latest})", fontsize=15, pad=15)

                # Plot Horizontal Bar with Error Bars (xerr)
                axes[0].barh(
                    latest_data['feature'],
                    latest_data['importance_mean'],
                    xerr=latest_data['importance_std'],
                    color="#1abc9c",
                    ecolor="#2c3e50", # Error bar color
                    capsize=4,        # Error bar caps
                    alpha=0.8
                )
                axes[0].set_xlabel("Mean Decrease in Error (Importance)")

                # Highlight the selected feature in the bar chart
                if selected_feature in latest_data['feature'].values:
                    idx = latest_data['feature'].tolist().index(selected_feature)
                    axes[0].get_yticklabels()[idx].set_color("#d35400")
                    axes[0].get_yticklabels()[idx].set_fontweight("bold")

                # --- PLOT 2: TIME TREND FOR SELECTED FEATURE ---
                feat_data = model_data[model_data['feature'] == selected_feature].sort_values('timestamp')

                if not feat_data.empty:
                    axes[1].set_title(f"Historical Trend & Stability: '{selected_feature}'\n({selected_model.upper()} Model)", fontsize=15, pad=15)

                    # Plot the Mean Line
                    axes[1].plot(
                        feat_data['timestamp'], feat_data['importance_mean'],
                        marker='o', linewidth=2.5, markersize=8, color="#d35400"
                    )

                    # Shade the Error Band (Mean - Std to Mean + Std)
                    axes[1].fill_between(
                        feat_data['timestamp'],
                        feat_data['importance_mean'] - feat_data['importance_std'],
                        feat_data['importance_mean'] + feat_data['importance_std'],
                        color="#d35400", alpha=0.15, edgecolor='none', label="±1 Std Dev (Error Band)"
                    )

                    axes[1].set_ylabel("Importance Score")

                    # Force clean X-axis ticks
                    unique_timestamps = feat_data['timestamp'].unique()
                    axes[1].set_xticks(unique_timestamps)
                    axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%b %d\n%H:00'))
                    axes[1].tick_params(axis='x', rotation=45)
                    axes[1].legend(loc="upper left", frameon=False)

                else:
                    axes[1].text(0.5, 0.5, f"No history for '{selected_feature}'", ha='center', fontsize=12)
                    axes[1].set_axis_off()

            else:
                for ax in axes:
                    ax.text(0.5, 0.5, "No Data Available", ha='center', fontsize=12)
                    ax.set_axis_off()

            plt.tight_layout(pad=3.0)
            plt.show()

    # ---------------------------------------------------------
    # 4. EVENT BINDING
    # ---------------------------------------------------------
    def on_model_change(change):
        # Update feature dropdown options when model changes
        new_model = change.new
        latest_ts = df_imp[df_imp['model'] == new_model]['timestamp'].max()
        new_feats = df_imp[(df_imp['model'] == new_model) & (df_imp['timestamp'] == latest_ts)]
        new_feats = new_feats.sort_values('importance_mean', ascending=False)['feature'].tolist()

        feature_dropdown.options = new_feats
        if new_feats:
            feature_dropdown.value = new_feats[0]
        update_plots()

    def on_feature_change(change):
        update_plots()

    model_dropdown.observe(on_model_change, names='value')
    feature_dropdown.observe(on_feature_change, names='value')

    controls = widgets.HBox([model_dropdown, feature_dropdown], layout=widgets.Layout(margin='0 0 20px 0'))
    dashboard_ui = widgets.VBox([widgets.HTML(header_html.data), controls, output_plot])

    display(dashboard_ui)
    update_plots()

# ==========================================
# Run the dashboard
# ==========================================
render_all_features_error_dashboard()

In [5]:
# @title 📈 Interactive PDP Time-Scrubber (Top 3 Features)
import os
import glob
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output

BASE_DIR = "/content/myscrapers-aya16102/results"
run_folders = sorted([f for f in os.listdir(BASE_DIR) if f.isdigit()])
pdp_data = {}

# ---------------------------------------------------------
# 1. Map all PDP PNGs
# ---------------------------------------------------------
for ts in run_folders:
    plot_dir = os.path.join(BASE_DIR, ts, "plots")
    if os.path.exists(plot_dir):
        # Look for any file matching *_pdp_*.png
        for img_path in glob.glob(os.path.join(plot_dir, "*_pdp_*.png")):
            filename = os.path.basename(img_path)

            # Extract the model and feature from the filename (e.g., xgb_pdp_age.png)
            parts = filename.replace(".png", "").split("_pdp_")
            if len(parts) == 2:
                model, feature = parts[0], parts[1]

                if model not in pdp_data:
                    pdp_data[model] = {}
                if ts not in pdp_data[model]:
                    pdp_data[model][ts] = []

                # Append to the list of plots for this model and timestamp
                pdp_data[model][ts].append((feature, img_path))

if not pdp_data:
    display(HTML("<b style='color:red;'>No PDP plots found.</b> Make sure the models successfully saved PDPs to the plots folder."))
else:
    # ---------------------------------------------------------
    # 2. Build Dashboard UI
    # ---------------------------------------------------------
    models_available = sorted(list(pdp_data.keys()))

    header_html = HTML("""
    <div style="font-family: Arial, sans-serif; margin-bottom: 20px; padding: 15px; background-color: #f8f9fa; border-radius: 8px; border-left: 5px solid #2980b9;">
        <h2 style="margin-top: 0; color: #2980b9;">📈 Partial Dependence Plot (PDP) Time-Scrubber</h2>
        <p style="margin-bottom: 5px; color: #2980b9;;">Use the slider to see how the <b>Top 3 Drivers of Price</b> and their effects shift over time.</p>
        <ul style="margin-top: 5px; font-size: 13px; color: #555;">
            <li>The X-Axis shows the value of the feature (e.g., Age of the car).</li>
            <li>The Y-Axis shows the expected impact on the predicted Price.</li>
        </ul>
    </div>
    """)

    pdp_model_dropdown = widgets.Dropdown(
        options=[(m.upper(), m) for m in models_available],
        value=models_available[0],
        description='<b>Model:</b>',
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='250px')
    )

    pdp_time_slider = widgets.SelectionSlider(
        options=run_folders,
        description='<b>Timeline:</b>',
        orientation='horizontal',
        readout=True,
        style={'description_width': 'initial'},
        layout=widgets.Layout(width='600px', margin='0 0 20px 0')
    )

    pdp_output_area = widgets.Output()

    # ---------------------------------------------------------
    # 3. Define UI Update Logic
    # ---------------------------------------------------------
    def update_pdp_plot(*args):
        model = pdp_model_dropdown.value
        ts = pdp_time_slider.value

        with pdp_output_area:
            clear_output(wait=True)
            if model in pdp_data and ts in pdp_data[model]:
                images = pdp_data[model][ts]
                formatted_date = pd.to_datetime(ts, format='%Y%m%d%H').strftime('%b %d, %Y - %H:00 UTC')

                display(HTML(f"<h3 style='color: #333;'>Top Features for {model.upper()} • {formatted_date}</h3>"))

                # Load images dynamically into an IPython Image widget
                img_widgets = []
                for feat, img_path in images:
                    with open(img_path, "rb") as f:
                        img_data = f.read()
                    # Set a fixed width so 3 images fit perfectly side-by-side
                    img_widgets.append(widgets.Image(value=img_data, format='png', width=400))

                # Display them side-by-side in a Horizontal Box
                display(widgets.HBox(img_widgets))
            else:
                display(HTML(f"<p style='color:gray;'>No PDP plots generated for <b>{model.upper()}</b> during run <b>{ts}</b>.</p>"))

    # ---------------------------------------------------------
    # 4. Bind and Render
    # ---------------------------------------------------------
    pdp_model_dropdown.observe(update_pdp_plot, names='value')
    pdp_time_slider.observe(update_pdp_plot, names='value')

    # Create the control row (Dropdown next to Slider)
    controls = widgets.HBox([pdp_model_dropdown, pdp_time_slider], layout=widgets.Layout(align_items='center'))

    display(widgets.VBox([widgets.HTML(header_html.data), controls, pdp_output_area]))
    update_pdp_plot() # Render first frame